# Os 16 rankings: pesos e métodos de agregação

Este notebook recalcula em Python os 16 rankings do desenho (soma ponderada, TOPSIS, VIKOR e PROMETHEE II × pesos iguais, CRITIC, AHP Gaussiano e AHP clássico), que antes estavam só nas planilhas de `planilhas/`. Os parâmetros são os do `docs/log_decisoes.md`, e as funções ficam em `mcda.py`, na mesma pasta.

Entrada: `planilhas/relatorio ABES 2026 NE.ods`, com os 505 municípios habilitados no ABES 2026 e os números já convertidos, a mesma base das planilhas. O notebook não grava arquivos; o notebook 03 recalcula os rankings com as mesmas funções.

As pontuações foram conferidas com as das planilhas em 9 nov. 2026: as 15 que não são a do ABES coincidem até a 15ª casa decimal (diferença máxima de 2 × 10⁻¹⁵), e a soma com pesos iguais reproduz a pontuação total do ABES (conferido abaixo).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import mcda

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir())
mun = mcda.ler_base(RAIZ)
X = mun[mcda.CRITERIOS].to_numpy(float)
print(X.shape[0], "municípios,", X.shape[1], "critérios")

505 municípios, 5 critérios


## 1. Pesos

Os pesos CRITIC e do AHP Gaussiano são recalculados a partir dos dados; os do AHP clássico, a partir da matriz de Saaty.

In [2]:
W = mcda.pesos(X)
w_saaty, rc = mcda.pesos_saaty()
tab_pesos = pd.DataFrame(W, index=mcda.NOMES)
print(f"RC da matriz de Saaty = {rc:.4f}")
(tab_pesos * 100).round(1)

RC da matriz de Saaty = 0.0152


,Pesos iguais,CRITIC,AHP Gaussiano,AHP clássico
Água,20.0,13.4,8.5,41.7
Esgoto,20.0,15.5,20.4,16.0
Tratamento,20.0,19.6,34.9,9.7
Coleta,20.0,14.9,6.8,26.3
Destinação,20.0,36.6,29.5,6.2


## 2. Pontuações e posições

Todas as pontuações estão no sentido "maior é melhor". No VIKOR, a coluna é −Q.

In [3]:
pont = mcda.rankings(X)
pos = pont.apply(mcda.posicoes)
dif = np.abs(pont["Soma ponderada | Pesos iguais"] - mun["Pontuação total"]).max()
print(f"Maior diferença entre a soma com pesos iguais e a pontuação do ABES: {dif:.2f}")
pos.loc[pos["Soma ponderada | Pesos iguais"] <= 10].assign(Município=mun["Município"]).set_index("Município").sort_values("Soma ponderada | Pesos iguais").iloc[:, :4]

Maior diferença entre a soma com pesos iguais e a pontuação do ABES: 0.00


,Soma ponderada | Pesos iguais,Soma ponderada | CRITIC,Soma ponderada | AHP Gaussiano,Soma ponderada | AHP clássico
Município,,,,
Salvador,1,1,1,1
Vitória da Conquista,2,2,7,2
União dos Palmares,3,4,2,3
Jequié,4,5,5,4
Porto Seguro,5,3,4,10
Campina Grande,6,6,13,9
Aracaju,7,7,19,5
Barra dos Coqueiros,8,8,21,7
Luís Eduardo Magalhães,9,9,15,8


## 3. Concordância entre os 16 rankings

Spearman ρ e Kendall τ-b entre todos os pares. A matriz de ρ reproduz a do parecer técnico.

In [4]:
ordem = [f"{m} | {e}" for e in mcda.ESQUEMAS for m in mcda.METODOS]
rho = pont[ordem].corr("spearman")
tau = pont[ordem].corr("kendall")
tau.round(2)

,Soma ponderada | Pesos iguais,TOPSIS | Pesos iguais,VIKOR | Pesos iguais,PROMETHEE II | Pesos iguais,Soma ponderada | CRITIC,TOPSIS | CRITIC,VIKOR | CRITIC,PROMETHEE II | CRITIC,Soma ponderada | AHP Gaussiano,TOPSIS | AHP Gaussiano,VIKOR | AHP Gaussiano,PROMETHEE II | AHP Gaussiano,Soma ponderada | AHP clássico,TOPSIS | AHP clássico,VIKOR | AHP clássico,PROMETHEE II | AHP clássico
Soma ponderada | Pesos iguais,1.00,0.76,0.90,0.82,0.86,0.78,0.80,0.92,0.71,0.66,0.72,0.83,0.71,0.52,0.53,0.53
TOPSIS | Pesos iguais,0.76,1.00,0.72,0.73,0.66,0.73,0.64,0.70,0.82,0.81,0.74,0.87,0.73,0.53,0.48,0.47
VIKOR | Pesos iguais,0.90,0.72,1.00,0.76,0.85,0.81,0.87,0.88,0.71,0.66,0.77,0.79,0.64,0.45,0.48,0.48
PROMETHEE II | Pesos iguais,0.82,0.73,0.76,1.00,0.70,0.61,0.64,0.79,0.61,0.58,0.61,0.73,0.83,0.66,0.67,0.69
Soma ponderada | CRITIC,0.86,0.66,0.85,0.70,1.00,0.86,0.94,0.91,0.67,0.61,0.70,0.76,0.58,0.38,0.41,0.41
TOPSIS | CRITIC,0.78,0.73,0.81,0.61,0.86,1.00,0.87,0.81,0.78,0.74,0.77,0.82,0.52,0.33,0.33,0.33
VIKOR | CRITIC,0.80,0.64,0.87,0.64,0.94,0.87,1.00,0.85,0.68,0.63,0.73,0.74,0.52,0.34,0.37,0.37
PROMETHEE II | CRITIC,0.92,0.70,0.88,0.79,0.91,0.81,0.85,1.00,0.68,0.63,0.70,0.79,0.65,0.46,0.49,0.50
Soma ponderada | AHP Gaussiano,0.71,0.82,0.71,0.61,0.67,0.78,0.68,0.68,1.00,0.92,0.86,0.86,0.58,0.38,0.35,0.33
TOPSIS | AHP Gaussiano,0.66,0.81,0.66,0.58,0.61,0.74,0.63,0.63,0.92,1.00,0.80,0.82,0.56,0.38,0.32,0.31


In [5]:
def resumo(mat):
    linhas = []
    for i, a in enumerate(ordem):
        for b in ordem[i + 1:]:
            ma, ea = a.split(" | "); mb, eb = b.split(" | ")
            tipo = ("Mesmos pesos, agregação diferente" if ea == eb else
                    "Mesma agregação, pesos diferentes" if ma == mb else "Pesos e agregação diferentes")
            linhas.append((tipo, mat.loc[a, b]))
    d = pd.DataFrame(linhas, columns=["Tipo de par", "valor"])
    return d.groupby("Tipo de par")["valor"].agg(pares="count", mínimo="min", máximo="max", média="mean").round(2)

print("Spearman ρ"); display(resumo(rho))
print("Kendall τ-b"); display(resumo(tau))
fora = tau.where(~np.eye(16, dtype=bool))
print(f"τ médio entre os 16 rankings: {fora.stack().mean():.2f}")

Spearman ρ


,pares,mínimo,máximo,média
Tipo de par,,,,
"Mesma agregação, pesos diferentes",24,0.45,0.95,0.78
"Mesmos pesos, agregação diferente",24,0.88,0.99,0.95
Pesos e agregação diferentes,72,0.44,0.99,0.75


Kendall τ-b


,pares,mínimo,máximo,média
Tipo de par,,,,
"Mesma agregação, pesos diferentes",24,0.33,0.87,0.63
"Mesmos pesos, agregação diferente",24,0.72,0.94,0.82
Pesos e agregação diferentes,72,0.31,0.92,0.60


τ médio entre os 16 rankings: 0.65


## 4. Mudanças de quartil em relação ao ABES

Quartil 1 = 25% melhores. Conta os municípios que mudam de quartil em cada ranking, em relação à soma ponderada com pesos iguais (o ABES).

In [6]:
quartil = pos.apply(lambda c: pd.qcut(c.rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int))
ref = quartil["Soma ponderada | Pesos iguais"]
muda = pd.DataFrame({
    "mudam de quartil": (quartil.ne(ref, axis=0)).sum(),
    "mudam 2+ quartis": (quartil.sub(ref, axis=0).abs() >= 2).sum(),
    "ρ com o ABES": rho["Soma ponderada | Pesos iguais"],
    "τ com o ABES": tau["Soma ponderada | Pesos iguais"],
}).loc[ordem]
muda.round(2)

,mudam de quartil,mudam 2+ quartis,ρ com o ABES,τ com o ABES
Soma ponderada | Pesos iguais,0,0,1.00,1.00
TOPSIS | Pesos iguais,148,12,0.91,0.76
VIKOR | Pesos iguais,79,1,0.98,0.90
PROMETHEE II | Pesos iguais,126,0,0.95,0.82
Soma ponderada | CRITIC,127,3,0.95,0.86
TOPSIS | CRITIC,155,5,0.93,0.78
VIKOR | CRITIC,152,22,0.92,0.80
PROMETHEE II | CRITIC,50,0,0.99,0.92
Soma ponderada | AHP Gaussiano,184,11,0.89,0.71
TOPSIS | AHP Gaussiano,208,21,0.84,0.66
